In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
file = next(RAW.glob("*berufe-heft-kldb2010*202610*.xlsx"))

raw = pd.read_excel(file, sheet_name="1.1", header=None, skiprows=15, nrows=6)
levels = raw.iloc[1:6, [2, 3, 11]].copy()      # столбцы: уровень, безработные, вакансии
levels.columns = ["level", "unemployed", "vacancies"]
levels["level"] = levels["level"].str.strip()
levels

,level,unemployed,vacancies
1,Helfer,1491232,112031
2,Fachkraft,826597,396673
3,Spezialist,229907,84469
4,Experte,240450,60800
5,keine Angabe,166412,20


In [2]:
levels["unemployed_per_vacancy"] = (levels.unemployed / levels.vacancies).round(1)
levels["share_unemployed"] = (levels.unemployed / levels.unemployed.sum() * 100).round(1)
levels["share_vacancies"] = (levels.vacancies / levels.vacancies.sum() * 100).round(1)
levels

,level,unemployed,vacancies,unemployed_per_vacancy,share_unemployed,share_vacancies
1,Helfer,1491232,112031,13.3,50.5,17.1
2,Fachkraft,826597,396673,2.1,28.0,60.7
3,Spezialist,229907,84469,2.7,7.8,12.9
4,Experte,240450,60800,4.0,8.1,9.3
5,keine Angabe,166412,20,8320.6,5.6,0.0


In [3]:
plot = levels[levels.level != "keine Angabe"].melt(
    id_vars="level", value_vars=["share_unemployed", "share_vacancies"],
    var_name="group", value_name="percent")
plot["group"] = plot["group"].map({"share_unemployed": "Unemployed", "share_vacancies": "Vacancies"})

fig = px.bar(plot, x="level", y="percent", color="group", barmode="group", text="percent",
             title="Half of the unemployed seek unskilled jobs — 60% of vacancies need skilled workers",
             labels={"level": "Skill level", "percent": "% of total"})
fig.show()

In [4]:
fig.write_html("../output/charts/01_mismatch_by_level.html")


## Finding 1: The mismatch is about skill level

- 50.5% of the unemployed look for **Helfer** (unskilled) jobs, but only 17.1% of vacancies are at this level → 13.3 unemployed per vacancy.
- 60.7% of vacancies require a **Fachkraft** (vocational training) → only 2.1 unemployed per vacancy.
- Caveat: only part of all vacancies are reported to the BA, especially for Experten.

Source: BA, Arbeitsmarkt nach Berufen, October 2026.